# Synthetic label sanity check

This notebook verifies the completed relevance and performance labels.

- **Relevance** asks whether a candidate should be considered for a job.
- **Performance** asks whether the candidate would succeed if they did the job.

Large Parquet files are read in batches, so all 17 million outcomes are checked without loading them all into memory.

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import json, os
os.environ.setdefault("MPLCONFIGDIR", "/tmp/marketplace-matplotlib")
import matplotlib.pyplot as plt
import pandas as pd
import pyarrow.parquet as pq
import seaborn as sns
from IPython.display import display
sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 80)
cwd=Path.cwd().resolve()
ROOT=next(p for p in [cwd,*cwd.parents] if (p/"ground-truth-generation").exists())
DATA=ROOT/"ground-truth-generation/data/v1"
CONFIG=json.loads((ROOT/"ground-truth-generation/config/v1.json").read_text())
MANIFEST=json.loads((DATA/"manifest.json").read_text())
print("Data directory:",DATA)
display(pd.Series(MANIFEST,name="value").to_frame())

## 1. Basic completeness

These checks catch missing entities, duplicate IDs or names, and broken joins to hidden tables.

In [ ]:
candidates=pd.read_parquet(DATA/"candidates.parquet")
ch=pd.read_parquet(DATA/"candidate_hidden.parquet")
jobs=pd.read_parquet(DATA/"jobs.parquet")
jh=pd.read_parquet(DATA/"job_hidden.parquet")
neg=pd.read_parquet(DATA/"relevance_negative_sample.parquet")
audit=pd.read_csv(DATA/"relevance_audit_sample.csv", dtype={"job_id": str, "candidate_id": str})
basic={
"Candidate count matches manifest":len(candidates)==MANIFEST["candidate_count"],
"Candidate IDs are unique":candidates.candidate_id.is_unique,
"Candidate names are unique and filled":candidates.name.is_unique and candidates.name.notna().all() and candidates.name.str.strip().ne("").all(),
"Candidate hidden IDs match":set(candidates.candidate_id)==set(ch.candidate_id),
"Job count matches manifest":len(jobs)==MANIFEST["job_count"],
"Job IDs are unique":jobs.job_id.is_unique,
"Job hidden IDs match":set(jobs.job_id)==set(jh.job_id)}
display(pd.DataFrame({"check":basic.keys(),"passed":basic.values()}))
assert all(basic.values())

## 2. Relevance labels

Cutoffs from the configuration:

- below **0.55**: irrelevant (`0`)
- **0.55** to below **0.72**: borderline (`1`)
- **0.72** or higher: clearly relevant (`2`)

Five grade-0 examples are stored per job. Other cross-family pairs are logically irrelevant and are not materialized.

In [ ]:
counts=Counter(); sums=defaultdict(lambda:defaultdict(float)); ns=Counter()
cols=["relevance_score","skill_score","occupation_score","seniority_score","experience_score","domain_score"]
thresholds_ok=True
for batch in pq.ParquetFile(DATA/"relevance_ground_truth.parquet").iter_batches(columns=["relevance_grade","relevant",*cols],batch_size=250_000):
 f=batch.to_pandas(); assert f.relevant.all() and f.relevance_score.between(0,1).all()
 thresholds_ok &= f.loc[f.relevance_grade.eq(1),"relevance_score"].between(CONFIG["thresholds"]["borderline"],CONFIG["thresholds"]["clearly_relevant"],inclusive="left").all()
 thresholds_ok &= f.loc[f.relevance_grade.eq(2),"relevance_score"].ge(CONFIG["thresholds"]["clearly_relevant"]).all()
 for g,x in f.groupby("relevance_grade"):
  g=int(g); counts[g]+=len(x); ns[g]+=len(x)
  for c in cols:sums[g][c]+=x[c].sum()
per_job=neg.groupby("job_id").size()
rel_checks={"Positive file contains grades 1 and 2":set(counts)=={1,2},"Scores obey grade thresholds":thresholds_ok,
"Every job has five negatives":len(per_job)==len(jobs) and per_job.min()==per_job.max()==5,
"Negative sample is grade 0":neg.relevance_grade.eq(0).all() and neg.relevant.eq(False).all(),
"Audit contains 200 of every grade":audit.groupby("relevance_grade").size().to_dict()=={0:200,1:200,2:200}}
display(pd.DataFrame({"check":rel_checks.keys(),"passed":rel_checks.values()}));assert all(rel_checks.values())
rel_counts=pd.Series({0:len(neg),**counts},name="stored_pairs").sort_index()
display(rel_counts.rename_axis("relevance_grade").to_frame())
ax=rel_counts.plot.bar(color=["#D55E00","#E69F00","#009E73"],figsize=(8,4),title="Stored relevance labels")
ax.set(xlabel="Relevance grade",ylabel="Pairs");ax.ticklabel_format(style="plain",axis="y");plt.xticks(rotation=0);plt.show()

In [ ]:
means=pd.DataFrame({g:{c:sums[g][c]/ns[g] for c in cols} for g in sorted(ns)}).T
means.index.name="relevance_grade";display(means.round(3))
means.plot.bar(figsize=(10,5));plt.title("Average compatibility components by grade");plt.ylabel("Average score");plt.xticks(rotation=0);plt.show()

## 3. Example pairs

These fixed audit rows support a quick human spot check.

In [ ]:
examples=(audit.groupby("relevance_grade",group_keys=False).head(5)
 .merge(jobs[["job_id","role_family","role_type","role_track","required_skills"]],on="job_id")
 .merge(candidates[["candidate_id","role_track","skills"]],on="candidate_id",suffixes=("_job","_candidate")))
display(examples[["relevance_grade","name","current_title","title","role_family","role_type","role_track_job","role_track_candidate","required_skills","skills","relevance_score"]])

## 4. Hidden simulator traits

These create uncertainty in performance. They are simulator truth and must not become model features.

In [ ]:
cc=["latent_ability","latent_reliability","latent_domain_depth","latent_skill_mastery"];jc=["latent_difficulty","latent_specialization"]
latent={"Candidate traits are in [0,1]":ch[cc].apply(lambda s:s.between(0,1).all()).all(),"Job traits are in [0,1]":jh[jc].apply(lambda s:s.between(0,1).all()).all(),"Candidate traits vary":ch[cc].std().gt(.05).all(),"Job traits vary":jh[jc].std().gt(.05).all()}
display(pd.DataFrame({"check":latent.keys(),"passed":latent.values()}));assert all(latent.values())
fig,ax=plt.subplots(1,2,figsize=(12,4));ch[cc].plot.hist(bins=30,alpha=.5,ax=ax[0],title="Hidden candidate traits");jh[jc].plot.hist(bins=30,alpha=.5,ax=ax[1],title="Hidden job traits");plt.tight_layout();plt.show()

## 5. Performance labels

Every outcome row is scanned. A deterministic sample is retained for the distribution chart.

In [ ]:
oc=Counter();ps=defaultdict(float);sc=Counter();pmin,pmax=1.,0.;equal=True;deferred=True;samples=[]
for i,b in enumerate(pq.ParquetFile(DATA/"marketplace_outcomes.parquet").iter_batches(columns=["relevance_grade","true_success_probability","potential_success","successful_performance","selected","observed_success"],batch_size=250_000)):
 f=b.to_pandas();assert f.true_success_probability.between(0,1).all();equal &= f.potential_success.equals(f.successful_performance);deferred &= f.selected.isna().all() and f.observed_success.isna().all();pmin=min(pmin,f.true_success_probability.min());pmax=max(pmax,f.true_success_probability.max());samples.append(f.sample(min(1000,len(f)),random_state=20260910+i))
 for g,x in f.groupby("relevance_grade"):g=int(g);oc[g]+=len(x);ps[g]+=x.true_success_probability.sum();sc[g]+=int(x.successful_performance.sum())
perf=pd.DataFrame({"pairs":pd.Series(oc),"mean_success_probability":pd.Series({g:ps[g]/oc[g] for g in oc}),"simulated_success_rate":pd.Series({g:sc[g]/oc[g] for g in oc})}).sort_index();perf.index.name="relevance_grade"
performance={"Outcome count matches manifest":sum(oc.values())==MANIFEST["outcome_pairs"],"Probabilities are valid":pmin>=0 and pmax<=1,"Performance label equals potential outcome":equal,"Observed labels are correctly deferred":deferred,"Mean probability rises with relevance":perf.mean_success_probability.is_monotonic_increasing,"Success rate rises with relevance":perf.simulated_success_rate.is_monotonic_increasing,"Probabilities are not extreme":pmin>.01 and pmax<.99,"Every grade has successes and failures":perf.simulated_success_rate.between(.05,.95).all()}
display(pd.DataFrame({"check":performance.keys(),"passed":performance.values()}));display(perf.style.format({"mean_success_probability":"{:.1%}","simulated_success_rate":"{:.1%}"}));print(f"Probability range: {pmin:.3f} to {pmax:.3f}");assert all(performance.values())

In [ ]:
sample=pd.concat(samples,ignore_index=True);fig,ax=plt.subplots(1,2,figsize=(12,4))
for g,x in sample.groupby("relevance_grade"):sns.kdeplot(x.true_success_probability,label=f"Grade {g}",ax=ax[0])
ax[0].set_title("Success probability distributions");ax[0].legend();perf[["mean_success_probability","simulated_success_rate"]].plot.bar(ax=ax[1],color=["#0072B2","#009E73"]);ax[1].set(title="Performance improves with relevance",ylabel="Rate",ylim=(0,.75));ax[1].tick_params(axis="x",rotation=0);plt.tight_layout();plt.show()

## 6. Conclusion

Automated consistency checks cannot replace manual judgment for ambiguous matches. The 600-row audit CSV is provided for that review.

In [ ]:
checks={**basic,**rel_checks,**latent,**performance};failed=[k for k,v in checks.items() if not v]
display(pd.DataFrame({"result":["PASS" if not failed else "FAIL"],"automated_checks":[len(checks)],"failed_checks":[len(failed)],"candidates":[len(candidates)],"jobs":[len(jobs)],"performance_pairs":[sum(oc.values())]}))
if failed:print("Failed:",failed)
else:
 print("The labeling pipeline passes all automated sanity checks.")
 print("Higher relevance grades have stronger compatibility and better simulated performance.")
 print("Every grade contains successes and failures, so performance is not a copy of relevance.")
 print("Remaining judgment step: review relevance_audit_sample.csv and fill reviewer_grade.")